# 22 — Neurons, Activations, and MLPs

**Master syllabus coverage:** V2 3.1–3.2

## Why this matters

The Transformer feed-forward sublayer is an MLP. Understanding affine maps, nonlinear gates, saturation, and hidden width is essential before assembling deeper networks.

## Learning objectives

- Implement stable common activation functions and inspect their derivatives.
- Explain why stacked affine layers need nonlinearity.
- Implement a two-layer MLP with explicit shape annotations.
- Use a hidden representation to solve a nonlinearly separable task.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A neuron is affine transformation plus nonlinearity

$$z=w\cdot x+b,\qquad h=\phi(z).$$

Stacking only affine maps is still one affine map. Nonlinear activations allow a network
to represent curved and piecewise decision boundaries. An MLP applies this per token in
a Transformer; attention handles communication between tokens.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0)

def sigmoid(x: np.ndarray) -> np.ndarray:
    # Stable branch avoids overflow for large negative x.
    positive = x >= 0
    result = np.empty_like(x, dtype=np.float64)
    result[positive] = 1 / (1 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    result[~positive] = exp_x / (1 + exp_x)
    return result

values = np.array([-1000.0, -2.0, 0.0, 2.0, 1000.0])
print("ReLU:", relu(values))
print("sigmoid:", sigmoid(values))
assert np.isfinite(sigmoid(values)).all()


## 2. Activation tradeoffs

- **Sigmoid/tanh:** smooth and bounded, but saturate and shrink gradients.
- **ReLU:** cheap and nonsaturating on positives, but zero gradient on negatives.
- **GELU:** smooth gating used in classic Transformers.
- **SiLU/Swish:** smooth self-gating used inside SwiGLU in many modern models.

There is no best activation independent of architecture, initialization, and budget.


In [ ]:
x = torch.linspace(-5, 5, 11, requires_grad=True)
activations = {
    "sigmoid": torch.sigmoid(x),
    "tanh": torch.tanh(x),
    "relu": F.relu(x),
    "gelu": F.gelu(x),
    "silu": F.silu(x),
}
for name, output in activations.items():
    gradient, = torch.autograd.grad(output.sum(), x, retain_graph=True)
    print(f"{name:7} output endpoints=({output[0]:.3f},{output[-1]:.3f}) "
          f"gradient endpoints=({gradient[0]:.3g},{gradient[-1]:.3g})")


## 3. Manual two-layer MLP

$$H=\phi(XW_1^\top+b_1),\qquad Y=HW_2^\top+b_2.$$

For token states $X=[B,T,C]$, a Transformer feed-forward network preserves `[B,T]`
while expanding features from $C$ to $C_{hidden}$ and projecting back to $C$.


In [ ]:
torch.manual_seed(42)
B, T, C, hidden = 2, 3, 4, 12
X = torch.randn(B, T, C)
W1, b1 = torch.randn(hidden, C), torch.randn(hidden)
W2, b2 = torch.randn(C, hidden), torch.randn(C)

H = F.gelu(X @ W1.T + b1)  # [B,T,hidden]
Y = H @ W2.T + b2          # [B,T,C]
assert H.shape == (B, T, hidden) and Y.shape == X.shape

module = torch.nn.Sequential(torch.nn.Linear(C, hidden), torch.nn.GELU(), torch.nn.Linear(hidden, C))
with torch.no_grad():
    module[0].weight.copy_(W1); module[0].bias.copy_(b1)
    module[2].weight.copy_(W2); module[2].bias.copy_(b2)
torch.testing.assert_close(Y, module(X))


## 4. Nonlinearity solves XOR

XOR is not linearly separable. A small MLP can bend the representation so a final linear
output separates it. This is a minimal demonstration of representation learning—not a
claim that every dataset needs a deep model.


In [ ]:
xor_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
xor_y = torch.tensor([0., 1., 1., 0.])
torch.manual_seed(1)
xor_model = torch.nn.Sequential(torch.nn.Linear(2, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1))
optimizer = torch.optim.Adam(xor_model.parameters(), lr=0.05)
for _ in range(500):
    logits = xor_model(xor_x).squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logits, xor_y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
predictions = (xor_model(xor_x).squeeze(-1) > 0).float()
print("XOR predictions:", predictions.tolist(), "loss:", loss.item())
assert torch.equal(predictions, xor_y)


## Failure modes to recognize

- **Saturated activation:** useful gradients approach zero.
- **Dead ReLU units:** preactivations remain negative and receive no gradient.
- **Missing nonlinearity:** multiple layers collapse algebraically to one affine map.
- **Output/target mismatch:** the final activation and loss duplicate or omit needed transformations.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Implement GELU's exact formula and compare it numerically with PyTorch.
2. Train XOR with no activation and document the irreducible failure.
3. Count MLP parameters for widths C and 4C, including biases, and apply it to C=768.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can implement an MLP forward pass, derive its shapes and parameter count, and explain the role of each activation.

**Next:** 23 — Scalar autograd.
